In [1]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import re
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

# Load datasets
macro_dataset_path = 'E:\\Economic_Data\\Input data\\Splitting Economic data\\Macro Dataset 1.csv'
merged_dataset_path = 'E:\\Economic_Data\\Input data\\merged-dataset\\merged_datasets.csv'
price_dataset_path = 'E:\\Economic_Data\\Input data\\price.csv'

macro_df = pd.read_csv(macro_dataset_path)
merged_df = pd.read_csv(merged_dataset_path)
price_df = pd.read_csv(price_dataset_path)

# Ensure datetime columns are in datetime format
merged_df['datetime'] = pd.to_datetime(merged_df['datetime'], errors='coerce')

# Drop rows where datetime conversion failed
merged_df.dropna(subset=['datetime'], inplace=True)

# Display the first few rows to verify changes
print(merged_df.head())

# Function to clean event names by extracting words and ignoring other parameters
def clean_event_name(event_name):
    return re.sub(r'\((?!WoW|YoY|MoM).*?\)', '', event_name).strip()

# Apply cleaning to event names in the merged dataset
merged_df['Cleaned_Event'] = merged_df['Event'].apply(clean_event_name)

# Display the cleaned event names
print("\nCleaned Event Names in Merged Dataset:")
print(merged_df['Cleaned_Event'])

# Filter the events that are in the macro dataset
filtered_merged_df = merged_df[merged_df['Cleaned_Event'].isin(macro_df['Event'])]

# Verify the number of unique cleaned events
unique_cleaned_events = filtered_merged_df['Cleaned_Event'].unique()
print(f"Number of unique events in macro dataset: {len(macro_df['Event'].unique())}")
print(f"Number of unique cleaned events in filtered merged dataset: {len(unique_cleaned_events)}")

# Group data by the cleaned event names
grouped_events_df = filtered_merged_df.groupby('Cleaned_Event')

# Display the first few groups to verify
for name, group in list(grouped_events_df)[:5]:
    print(f"Group name: {name}")
    print(group.head())

# Create a deep copy of the filtered DataFrame to avoid SettingWithCopyWarning
filtered_merged_df = filtered_merged_df.copy()

# Convert columns to string type before cleaning
filtered_merged_df['Actual'] = filtered_merged_df['Actual'].astype(str)
filtered_merged_df['Forecast'] = filtered_merged_df['Forecast'].astype(str)
filtered_merged_df['Previous'] = filtered_merged_df['Previous'].astype(str)

# Function to clean numeric columns
def clean_numeric_column(column):
    return pd.to_numeric(
        column.str.replace('%', '')
               .str.replace('M', 'e6')
               .str.replace('B', 'e9')
               .str.replace('K', 'e3')
               .str.replace('[^\d.e]', '', regex=True),  # Remove any non-numeric and non-decimal characters
        errors='coerce'
    ).fillna(0)

# Apply cleaning to the Actual, Forecast, and Previous columns
filtered_merged_df['Actual'] = clean_numeric_column(filtered_merged_df['Actual'])
filtered_merged_df['Forecast'] = clean_numeric_column(filtered_merged_df['Forecast'])
filtered_merged_df['Previous'] = clean_numeric_column(filtered_merged_df['Previous'])

# Display the cleaned columns to verify
print("\nCleaned Numeric Columns:")
print(filtered_merged_df[['Actual', 'Forecast', 'Previous']])

# Calculate differences
filtered_merged_df['Actual_Previous_Diff'] = filtered_merged_df['Actual'] - filtered_merged_df['Previous']
filtered_merged_df['Actual_Forecast_Diff'] = filtered_merged_df['Actual'] - filtered_merged_df['Forecast']

# Ensure differences are numeric
filtered_merged_df['Actual_Previous_Diff'] = pd.to_numeric(filtered_merged_df['Actual_Previous_Diff'], errors='coerce')
filtered_merged_df['Actual_Forecast_Diff'] = pd.to_numeric(filtered_merged_df['Actual_Forecast_Diff'], errors='coerce')

# Display the data types to verify
print("\nData types after converting differences to numeric:")
print(filtered_merged_df.dtypes)

# Combine the differences from all events
all_differences = filtered_merged_df[['Cleaned_Event', 'Actual_Previous_Diff', 'Actual_Forecast_Diff']]

# Function to calculate VIF for a given dataframe
def calculate_vif(data):
    features = data.columns
    vif_data = pd.DataFrame()
    vif_data['feature'] = features
    vif_data['VIF'] = [variance_inflation_factor(data.values, i)
                       for i in range(len(features))]
    return vif_data

# Perform VIF calculation on combined differences
features = ['Actual_Previous_Diff', 'Actual_Forecast_Diff']
vif_results = calculate_vif(all_differences[features])

# Display the VIF results
print("\nVIF Results Across Events:")
print(vif_results)

price_df['datetime'] = pd.to_datetime(price_df['datetime'])

# Merge datasets on datetime
merged_data = pd.merge_asof(filtered_merged_df.sort_values('datetime'), 
                            price_df[['datetime', 'close']].sort_values('datetime'), 
                            on='datetime', 
                            direction='backward')

# Display the merged data to verify
print("\nMerged Data:")
print(merged_data.head())

# Function to calculate percentage change
def calculate_percentage_change(current_price, future_price):
    return ((future_price - current_price) / current_price) * 100

# Merge close prices at 5, 15, 30, and 60 minutes later
time_deltas = [5, 15, 30, 60]
for delta in time_deltas:
    future_price_df = price_df[['datetime', 'close']].copy()
    future_price_df['datetime'] = future_price_df['datetime'] - pd.Timedelta(minutes=delta)
    future_price_df.rename(columns={'close': f'close_price_{delta}m'}, inplace=True)
    merged_data = pd.merge_asof(merged_data.sort_values('datetime'), 
                                future_price_df.sort_values('datetime'), 
                                on='datetime', 
                                direction='forward')

# Calculate percentage changes
for delta in time_deltas:
    merged_data[f'pct_change_{delta}m'] = calculate_percentage_change(merged_data['close'], merged_data[f'close_price_{delta}m'])

# Display the data with percentage changes to verify
print("\nData with Percentage Changes:")
print(merged_data.head())

# Function to perform Ridge Regression analysis with PCA
def pca_ridge_regression_analysis(event_data, y_column):
    X = event_data[['Actual_Previous_Diff', 'Actual_Forecast_Diff']]
    y = event_data[y_column]

    # Create a pipeline with standard scaler, PCA, and ridge regression
    pca_ridge_model = make_pipeline(StandardScaler(), PCA(n_components=2), Ridge(alpha=1.0))  # alpha is the regularization strength

    # Fit the model
    pca_ridge_model.fit(X, y)

    # Get the coefficients
    pca = pca_ridge_model.named_steps['pca']
    ridge = pca_ridge_model.named_steps['ridge']
    coefficients = ridge.coef_
    intercept = ridge.intercept_
    r_squared = pca_ridge_model.score(X, y)

    return coefficients, intercept, r_squared

# Loop through each group and perform PCA Ridge Regression analysis for each time delta
results = []

for event, event_data in merged_data.groupby('Cleaned_Event'):
    print(f"Processing event: {event}")  # Debug print

    for delta in time_deltas:
        y_column = f'pct_change_{delta}m'
        
        # Drop rows with NaN values in the dependent or independent variables
        valid_data = event_data.dropna(subset=['Actual_Previous_Diff', 'Actual_Forecast_Diff', y_column])

        if not valid_data.empty:
            # Perform PCA Ridge Regression analysis
            coefficients, intercept, r_squared = pca_ridge_regression_analysis(valid_data, y_column)
            result = {
                'Event': event,
                'Time_Delta': delta,
                'Coefficients': coefficients,
                'Intercept': intercept,
                'R_squared': r_squared
            }
            results.append(result)

# Convert results to DataFrame for easier viewing
results_df = pd.DataFrame(results)

# Save results to a CSV file
results_file_path = 'E:\\Economic_Data\\Output data\\modelling output\\pca_ridge_regression_results.csv'
results_df.to_csv(results_file_path, index=False)

print(f"Results saved to {results_file_path}")


  Cur.  Imp.                                  Event   Actual Forecast  \
0  USD   3.0  S&P Global US Manufacturing PMI (Dec)     46.2     46.2   
1  USD   2.0      Construction Spending (MoM) (Nov)    0.20%   -0.40%   
2  USD   1.0                   3-Month Bill Auction    4.41%      NaN   
3  USD   1.0                   6-Month Bill Auction    4.64%      NaN   
4  USD   NaN        MBA Mortgage Applications (WoW)  -10.30%      NaN   

  Previous            datetime  
0     47.7 2023-01-03 14:45:00  
1   -0.20% 2023-01-03 15:00:00  
2    4.35% 2023-01-03 16:30:00  
3    4.60% 2023-01-03 16:30:00  
4    0.90% 2023-01-04 12:00:00  

Cleaned Event Names in Merged Dataset:
0              S&P Global US Manufacturing PMI
1                  Construction Spending (MoM)
2                         3-Month Bill Auction
3                         6-Month Bill Auction
4              MBA Mortgage Applications (WoW)
                         ...                  
5637             OPEC Crude oil Productio